In [0]:
# Databricks notebook source
# ============================================================================
# 03. GOLD LAYER - MARGIN COMPUTATION & QUARANTINE ROUTING
# Objective: Process 20k items, quarantine anomalies, and optimize table.
# ============================================================================

from pyspark.sql.functions import col, round as spark_round, current_timestamp, when

SILVER_TABLE = "default.silver_cpq_quote_items"
GOLD_CLEAN_TABLE = "default.gold_manufacturing_margins"
GOLD_ANOMALY_TABLE = "default.gold_margin_anomalies"

# COMMAND ----------
df_silver = spark.table(SILVER_TABLE)

# 1. Compute Financial KPIs & Margin %
df_computed = df_silver.withColumn("total_revenue", spark_round(col("net_price").cast('double') * col("quantity").cast('double'), 2)) \
                       .withColumn("total_cost", spark_round(col("unit_cost").cast('double') * col("quantity").cast('double'), 2)) \
                       .withColumn("margin_amount", spark_round(col("total_revenue") - col("total_cost"), 2)) \
                       .withColumn("margin_pct", 
                           when(col("total_revenue") <= 0, -999.99) # Safeguard against zero revenue division
                          .otherwise(spark_round((col("margin_amount") / col("total_revenue")) * 100, 2))
                       ) \
                       .withColumn("margin_status", 
                            when(col("margin_pct") < 15.0, "Erosion Alert")
                           .when(col("margin_pct").between(15.0, 30.0), "Healthy")
                           .otherwise("High Margin")
                       )

# COMMAND ----------
# 2. Split Data Stream: Clean vs. Quarantine (Threshold: -50% to 95% Margin)
is_valid_margin = (col("margin_pct") >= -50.0) & (col("margin_pct") <= 95.0)

df_clean = df_computed.filter(is_valid_margin)

df_anomalies = df_computed.filter(~is_valid_margin) \
                          .withColumn("flagged_timestamp", current_timestamp()) \
                          .withColumn("anomaly_reason", 
                              when(col("margin_pct") == -999.99, "Invalid/Zero Revenue")
                             .when(col("margin_pct") < -50.0, "Negative Margin Exceeds Threshold")
                             .otherwise("Unusually High Margin (> 95%)")
                          )

# COMMAND ----------
# 3. Write Clean & Anomalous Records to Delta Tables
df_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_CLEAN_TABLE)
df_anomalies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_ANOMALY_TABLE)

# COMMAND ----------
# 4. Performance Tuning: Delta Lake Z-Ordering for Fast Downstream BI
spark.sql(f"OPTIMIZE {GOLD_CLEAN_TABLE} ZORDER BY (quote_date, product_category, margin_status)")

# COMMAND ----------
# 5. Output Batch Summary Logs
clean_cnt = df_clean.count()
anomaly_cnt = df_anomalies.count()

print("==========================================================")
print(f"📊 PIPELINE EXECUTION SUMMARY FOR {20000:,} RECORDS")
print("==========================================================")
print(f" Clean Records Written to Gold Table:     {clean_cnt:,}")
print(f"⚠️  Anomalous Records Routed to Quarantine: {anomaly_cnt:,}")
print("==========================================================")